In [ ]:
# Imports
import torch
import datasets
# import evaluate
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from transformers import TrainingArguments, Trainer
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

In [ ]:
# Load Assamese dataset
from datasets import load_dataset, Audio

common_voice = load_dataset("mozilla-foundation/common_voice_11_0", "as", split={"train": "train", "test": "test[:10]"},trust_remote_code=True)

In [ ]:
common_voice

In [ ]:
common_voice['train'][0]

In [ ]:
# Resample audio to 16kHz
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
# Load Whisper Feature Extractor, Tokenizer, Processor
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="assamese", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="assamese", task="transcribe")

In [ ]:
# Preprocessing Function
def prepare_dataset(batch):
    audio = batch["audio"]

    # Extract features
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=16000).input_features[0]

    # Tokenize transcription
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

# Apply preprocessing to train and test splits
common_voice["train"] = common_voice["train"].map(prepare_dataset, remove_columns=common_voice["train"].column_names)
common_voice["test"] = common_voice["test"].map(prepare_dataset, remove_columns=common_voice["test"].column_names)

In [ ]:
# Load Whisper-Tiny model
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

# Disable forced decoder IDs
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
# Step 8: Define Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
# Load WER metric
wer_metric = evaluate.load("wer")

In [ ]:
# Step 10: Define Compute Metrics
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-assamese",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    evaluation_strategy="steps",
    logging_strategy="steps",
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    max_steps=100,
    warmup_steps=50,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

In [ ]:
# Initialize Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
)

In [ ]:
# Training
trainer.train()